In [1]:
from pathlib import Path

import anndata
from anndata import AnnData
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import RocCurveDisplay

import zs_perturbation
from zs_perturbation import BENCHMARK_FILE, CONTROL, DATASET_NAME, TO_GENE_SYMBOL, DISEASES

Loading weights:   0%|          | 0/319 [00:00<?, ?it/s]

In [2]:
zs_perturbation.download_dataset()

Dataset 'bulk-rna-immuno-inflammation-cohorts' already exists. Skipping download.


In [3]:
disease = DISEASES[0]
disease

'AD'

In [4]:
adata = zs_perturbation.load_dataset(disease)

/Users/quentinblampey/dev/zs_perturbation/zs_perturbation/io.py:26: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3")


In [5]:
adata_control = adata[adata.obs["disease"] == CONTROL][:10].copy()
adata_disease = adata[adata.obs["disease"] != CONTROL][:10].copy()

In [17]:
zs_perturbation.register_hook()

In [18]:
zs_perturbation.store_z_intermediate(adata_control)

Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
healthy_centroid = adata_control.obsm["z_intermediate"].mean(axis=0)

In [7]:
from zs_perturbation.benchmark import DF_BENCH

df_disease = DF_BENCH[DF_BENCH["disease_abbrev"] == disease]

In [ ]:
genes = [
    TO_GENE_SYMBOL.get(gene, gene) for gene in df_disease["target_genes"].str.split(";").explode().unique() if gene
]

In [15]:
len(genes)

23

In [ ]:
zs_perturbation.compute_healthy_score(adata_disease, healthy_centroid, genes)

Processing gene TNFSF4 (1/23)


Processing batches:   0%|          | 0/1 [00:00<?, ?it/s]

Processing gene TNFSF13B (2/23)


/Users/quentinblampey/dev/zs_perturbation/zs_perturbation/decoder_based.py:74: RuntimeWarning: invalid value encountered in divide
  healthy_direction /= np.linalg.norm(healthy_direction)


Processing batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing batches:   0%|          | 0/1 [00:00<?, ?it/s]

Processing gene IL33 (8/23)


/Users/quentinblampey/dev/zs_perturbation/zs_perturbation/decoder_based.py:74: RuntimeWarning: invalid value encountered in divide
  healthy_direction /= np.linalg.norm(healthy_direction)


Processing batches:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [6]:
from zs_perturbation.eva import model, tokenizer

In [7]:
z_holder = {}


def hook_fn(module, _, output):
    z_holder["z"] = output
    output.retain_grad()


handle = model.layers[-2].register_forward_hook(hook_fn)

In [8]:
import torch
import numpy as np

device = "cpu"

token_ids = tokenizer.convert_tokens_to_ids(adata.var_names)
token_ids_tensor = torch.tensor(token_ids, dtype=torch.long, device=device)

batch_adata = adata[:1]

batch_genes = token_ids_tensor.unsqueeze(0).expand(batch_adata.n_obs, -1)
batch_values = torch.from_numpy(batch_adata.X.astype(np.float32)).to(device)

output = model.encode(gene_ids=batch_genes, expression_values=batch_values)

In [9]:
predicted_expression = model.decode(output.gene_embeddings)

In [10]:
i = 13

loss = predicted_expression[:, i].mean()

In [11]:
loss.backward()

In [12]:
z_holder["z"].grad.shape

torch.Size([1, 2027, 256])

In [13]:
adata_disease.obsm["z_intermediate"] = np.zeros((adata_disease.n_obs, 256))
adata_disease.obsm["grad"] = np.zeros((adata_disease.n_obs, 256))

In [14]:
import numpy as np
import torch
from anndata import AnnData
from tqdm.auto import tqdm


def encode(adata: AnnData, device: str = "cpu", batch_size: int = 25) -> None:
    adata.obsm["z_intermediate"] = np.zeros((adata.n_obs, 256))

    token_ids = tokenizer.convert_tokens_to_ids(adata.var_names)
    token_ids_tensor = torch.tensor(token_ids, dtype=torch.long, device=device)

    for i in tqdm(range(0, adata.n_obs, batch_size), desc="Encoding"):
        batch_adata = adata[i : i + batch_size]

        batch_genes = token_ids_tensor.unsqueeze(0).expand(batch_adata.n_obs, -1)
        batch_values = torch.from_numpy(batch_adata.X.astype(np.float32)).to(device)

        model.encode(gene_ids=batch_genes, expression_values=batch_values)

        adata.obsm["z_intermediate"][i : i + batch_size] = z_holder["z"][:, 0].numpy(force=True)

In [15]:
encode(adata_control)

Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import numpy as np
import torch
from anndata import AnnData
from tqdm.auto import tqdm

adata_disease.var["mean_healthy_score"] = 0.0


def compute_healthy_score(adata: AnnData, gene: str, device: str = "cpu", batch_size: int = 25) -> None:
    adata.obsm["z_intermediate"] = np.zeros((adata.n_obs, 256))
    adata.obsm["grad"] = np.zeros((adata.n_obs, 256))

    token_ids = tokenizer.convert_tokens_to_ids(adata.var_names)
    token_ids_tensor = torch.tensor(token_ids, dtype=torch.long, device=device)

    for i in tqdm(range(0, adata.n_obs, batch_size), desc="Encoding"):
        batch_adata = adata[i : i + batch_size]

        batch_genes = token_ids_tensor.unsqueeze(0).expand(batch_adata.n_obs, -1)
        batch_values = torch.from_numpy(batch_adata.X.astype(np.float32)).to(device)

        output = model.encode(gene_ids=batch_genes, expression_values=batch_values)

        predicted_expression = model.decode(output.gene_embeddings)

        loss = predicted_expression[:, i].mean()
        loss.backward()

        adata.obsm["z_intermediate"][i : i + batch_size] = z_holder["z"][:, 0].numpy(force=True)

        grad = z_holder["z"].grad[:, 0].numpy(force=True)
        adata.obsm["grad"][i : i + batch_size] = grad / np.linalg.norm(grad, axis=1, keepdims=True)
        z_holder["z"].grad.zero_()

    healthy_direction = adata_control.obsm["z_intermediate"].mean(0) - adata_disease.obsm["z_intermediate"].mean(0)
    healthy_direction /= np.linalg.norm(healthy_direction)

    adata_disease.obs["healthy_score"] = adata_disease.obsm["grad"] @ healthy_direction

    to_entrez_id = pd.Series(adata.var_names, index=adata.var["gene_symbols"])
    entrez_id = to_entrez_id[gene]
    adata.var.loc[entrez_id, "mean_healthy_score"] = adata_disease.obs["healthy_score"].mean()

In [25]:
compute_healthy_score(adata_disease, gene="IL17A")

Encoding:   0%|          | 0/1 [00:00<?, ?it/s]

/var/folders/rl/nsddz37s55zbfg5h7b060rlc0000gn/T/ipykernel_8177/13975457.py:42: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.02785692574062018' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  adata.var.loc[entrez_id, "mean_healthy_score"] = adata_disease.obs["healthy_score"].mean()
